# 第16章 高级系统软件 - 操作系统高级功能
# Chapter 16 Advanced System Software - Advanced OS Features

## Cambridge A-Level Computer Science (9618) - A2 Paper 3

---

### 本节学习目标 (Learning Objectives)

完成本notebook后，你将能够：

1. **进程管理 (Process Management)**: 理解进程状态转换、各种调度算法 (Scheduling Algorithms)
2. **内存管理 (Memory Management)**: 掌握分页 (Paging)、分段 (Segmentation)、虚拟内存 (Virtual Memory) 和页面置换算法
3. **死锁 (Deadlock)**: 了解死锁的条件、预防、避免和检测
4. **文件系统 (File Systems)**: 比较 FAT、NTFS、ext4 文件系统

---

### 前置知识 (Prerequisites)

- 基本的操作系统概念（第4章）
- Python 编程基础
- 队列 (Queue) 和栈 (Stack) 数据结构

In [ ]:
# 环境配置 (Environment Setup)
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import deque
import copy

# 中文字体配置 - 确保图表能正确显示中文
matplotlib.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

print("环境配置完成！(Environment setup complete!)")

---
## 第一部分：进程管理 (Process Management)

### 1.1 什么是进程？(What is a Process?)

**进程 (Process)** 是一个正在执行的程序的实例。它不仅仅是程序代码本身，还包括：

| 组成部分 | 英文 | 说明 |
|---------|------|------|
| 程序代码 | Program Code | 即 text section，存放可执行指令 |
| 程序计数器 | Program Counter (PC) | 指向下一条要执行的指令 |
| 寄存器内容 | Register Contents | CPU 寄存器的当前值 |
| 栈 | Stack | 存放临时数据（函数参数、返回地址、局部变量） |
| 数据段 | Data Section | 全局变量 |
| 堆 | Heap | 运行时动态分配的内存 |

### 进程 vs 程序 (Process vs Program)

- **程序 (Program)**: 存储在磁盘上的被动实体 (passive entity)，是一组指令的集合
- **进程 (Process)**: 加载到内存中的主动实体 (active entity)，拥有程序计数器和关联资源

> **考试重点**: 一个程序可以产生多个进程。例如多个用户同时运行同一个文字处理程序。

### 1.2 进程状态 (Process States)

每个进程在其生命周期中会经历不同的状态。经典的五状态模型 (Five-State Model):

| 状态 | 英文 | 说明 |
|------|------|------|
| 新建 | **New** | 进程刚被创建，尚未加入就绪队列 |
| 就绪 | **Ready** | 进程已准备好运行，等待 CPU 分配 |
| 运行 | **Running** | 进程正在 CPU 上执行 |
| 等待/阻塞 | **Waiting/Blocked** | 进程在等待某个事件（如 I/O 完成） |
| 终止 | **Terminated** | 进程执行完毕或被终止 |

### 状态转换 (State Transitions)

- **New → Ready**: 进程被接纳进入系统 (Admitted)
- **Ready → Running**: 调度程序选中该进程 (Scheduler Dispatch)
- **Running → Ready**: 时间片用完或被高优先级进程抢占 (Timeout/Preempted)
- **Running → Waiting**: 进程请求 I/O 或等待事件 (I/O or Event Wait)
- **Waiting → Ready**: I/O 完成或等待的事件发生 (I/O or Event Completion)
- **Running → Terminated**: 进程执行完毕或出错 (Exit)

In [ ]:
# 可视化进程状态转换图 (Process State Transition Diagram)

fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('进程状态转换图 (Process State Transition Diagram)', fontsize=16, fontweight='bold', pad=20)

# 定义状态位置和颜色
states = {
    'New':        (2, 8, '#AED6F1', '新建\nNew'),
    'Ready':      (5, 5, '#A9DFBF', '就绪\nReady'),
    'Running':    (9, 5, '#F9E79F', '运行\nRunning'),
    'Waiting':    (7, 1.5, '#F5B7B1', '等待/阻塞\nWaiting'),
    'Terminated': (12, 8, '#D7BDE2', '终止\nTerminated'),
}

# 绘制状态圆
for name, (x, y, color, label) in states.items():
    circle = plt.Circle((x, y), 1.2, color=color, ec='black', lw=2, zorder=2)
    ax.add_patch(circle)
    ax.text(x, y, label, ha='center', va='center', fontsize=10, fontweight='bold', zorder=3)

# 绘制箭头转换
transitions = [
    ('New', 'Ready', '接纳 Admitted', (3.0, 7.2), (4.2, 5.9), (2.5, 6.5)),
    ('Ready', 'Running', '调度 Dispatch', (6.2, 5), (7.8, 5), (7, 5.5)),
    ('Running', 'Ready', '超时 Timeout', (7.8, 5.8), (6.2, 5.8), (7, 6.6)),
    ('Running', 'Waiting', 'I/O等待', (8.5, 3.9), (7.7, 2.6), (9, 2.8)),
    ('Waiting', 'Ready', 'I/O完成', (5.9, 2.0), (4.6, 4.0), (4.2, 2.5)),
    ('Running', 'Terminated', '退出 Exit', (10, 6.0), (11.2, 7.2), (11.2, 6.2)),
]

for src, dst, label, start, end, text_pos in transitions:
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle='->', lw=2, color='#333333'))
    ax.text(text_pos[0], text_pos[1], label, fontsize=9, ha='center', va='center',
            color='#8B0000', fontstyle='italic',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor='gray'))

plt.tight_layout()
plt.show()
print("\n上图展示了进程在五个状态之间的转换关系。")
print("考试中经常要求画出此图并解释每个转换的原因。")

### 1.3 进程控制块 PCB (Process Control Block)

操作系统用 **进程控制块 (PCB)** 来管理每个进程的信息。PCB 是操作系统中最重要的数据结构之一。

| PCB 字段 | 英文 | 说明 |
|----------|------|------|
| 进程ID | Process ID (PID) | 唯一标识一个进程 |
| 进程状态 | Process State | New/Ready/Running/Waiting/Terminated |
| 程序计数器 | Program Counter | 下一条指令的地址 |
| CPU寄存器 | CPU Registers | 所有寄存器的值 |
| CPU调度信息 | CPU Scheduling Info | 优先级、调度队列指针等 |
| 内存管理信息 | Memory Management Info | 页表/段表、基址/界限寄存器等 |
| I/O状态信息 | I/O Status Info | 已分配的I/O设备、打开的文件列表 |
| 记账信息 | Accounting Info | CPU使用时间、时间限制等 |

当发生 **上下文切换 (Context Switch)** 时，操作系统需要：
1. 保存当前运行进程的 PCB（保存现场）
2. 加载新进程的 PCB（恢复现场）

> **考试重点**: 上下文切换本身是一种 overhead（开销），因为在切换过程中 CPU 不做任何有用的工作。

### 1.4 CPU 调度算法 (Scheduling Algorithms)

**调度程序 (Scheduler)** 的任务是决定下一个运行的进程。评价调度算法的标准：

| 指标 | 英文 | 目标 | 说明 |
|------|------|------|------|
| CPU利用率 | CPU Utilization | 越高越好 | CPU 忙碌的时间比例 |
| 吞吐量 | Throughput | 越大越好 | 单位时间完成的进程数 |
| 周转时间 | Turnaround Time | 越短越好 | 从提交到完成的总时间 |
| 等待时间 | Waiting Time | 越短越好 | 在就绪队列中等待的总时间 |
| 响应时间 | Response Time | 越短越好 | 从提交到首次响应的时间 |

**关键公式**:
- **周转时间 (Turnaround Time)** = 完成时间 - 到达时间
- **等待时间 (Waiting Time)** = 周转时间 - 执行时间（CPU burst time）

下面我们逐一学习五种调度算法。

In [ ]:
# ========================================
# 调度算法模拟器 - 数据准备
# ========================================

class Process:
    """表示一个进程的类"""
    def __init__(self, pid, arrival_time, burst_time, priority=0):
        self.pid = pid                    # 进程ID
        self.arrival_time = arrival_time  # 到达时间
        self.burst_time = burst_time      # CPU执行时间
        self.priority = priority          # 优先级（数字越小优先级越高）
        self.remaining_time = burst_time  # 剩余执行时间
        self.start_time = -1              # 首次开始执行的时间
        self.finish_time = 0              # 完成时间
        self.waiting_time = 0             # 等待时间
    
    def __repr__(self):
        return f"P{self.pid}(到达={self.arrival_time}, 执行={self.burst_time}, 优先级={self.priority})"

# 示例进程数据
# 我们将使用这组数据来演示所有算法
def create_sample_processes():
    """创建示例进程集合"""
    return [
        Process('A', arrival_time=0, burst_time=6, priority=3),
        Process('B', arrival_time=1, burst_time=4, priority=1),
        Process('C', arrival_time=2, burst_time=2, priority=4),
        Process('D', arrival_time=3, burst_time=3, priority=2),
    ]

# 显示进程信息
processes = create_sample_processes()
print("示例进程数据 (Sample Process Data):")
print("=" * 60)
print(f"{'进程PID':^10}{'到达时间':^12}{'执行时间':^12}{'优先级':^10}")
print("-" * 60)
for p in processes:
    print(f"{'P'+str(p.pid):^10}{p.arrival_time:^12}{p.burst_time:^12}{p.priority:^10}")
print("=" * 60)
print("注：优先级数字越小 = 优先级越高 (lower number = higher priority)")

In [ ]:
# ========================================
# Gantt图绘制函数
# ========================================

def draw_gantt_chart(schedule, title, processes_info=None):
    """
    绘制Gantt图
    schedule: [(进程名, 开始时间, 结束时间), ...]
    title: 图表标题
    processes_info: 进程统计信息字典
    """
    colors = {'A': '#FF9999', 'B': '#66B2FF', 'C': '#99FF99', 'D': '#FFCC99', 
              'E': '#FF99FF', 'Idle': '#DDDDDD'}
    
    fig, axes = plt.subplots(2 if processes_info else 1, 1, 
                             figsize=(14, 6 if processes_info else 3),
                             gridspec_kw={'height_ratios': [1, 1] if processes_info else [1]})
    
    ax = axes[0] if processes_info else axes
    
    max_time = max(end for _, _, end in schedule)
    ax.set_xlim(0, max_time)
    ax.set_ylim(0, 1)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('时间 (Time Units)')
    ax.set_yticks([])
    ax.set_xticks(range(max_time + 1))
    
    for proc_name, start, end in schedule:
        color = colors.get(proc_name, '#CCCCCC')
        ax.barh(0.5, end - start, left=start, height=0.6, 
                color=color, edgecolor='black', linewidth=1.5)
        if end - start > 0.3:
            ax.text((start + end) / 2, 0.5, f'P{proc_name}', 
                    ha='center', va='center', fontsize=11, fontweight='bold')
    
    # 添加图例
    legend_patches = [mpatches.Patch(color=colors.get(p, '#CCC'), label=f'P{p}') 
                      for p in sorted(set(name for name, _, _ in schedule)) if p != 'Idle']
    ax.legend(handles=legend_patches, loc='upper right', fontsize=9)
    
    # 绘制统计信息表格
    if processes_info:
        ax2 = axes[1]
        ax2.axis('off')
        
        table_data = [['进程', '到达时间', '执行时间', '完成时间', '周转时间', '等待时间']]
        for pid, info in sorted(processes_info.items()):
            turnaround = info['finish'] - info['arrival']
            waiting = turnaround - info['burst']
            table_data.append([f'P{pid}', info['arrival'], info['burst'], 
                              info['finish'], turnaround, waiting])
        
        # 计算平均值
        avg_turnaround = np.mean([info['finish'] - info['arrival'] for info in processes_info.values()])
        avg_waiting = np.mean([info['finish'] - info['arrival'] - info['burst'] for info in processes_info.values()])
        table_data.append(['平均', '', '', '', f'{avg_turnaround:.2f}', f'{avg_waiting:.2f}'])
        
        table = ax2.table(cellText=table_data, loc='center', cellLoc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 1.5)
        
        # 表头加色
        for j in range(6):
            table[0, j].set_facecolor('#4472C4')
            table[0, j].set_text_props(color='white', fontweight='bold')
        # 最后一行（平均）加色
        for j in range(6):
            table[len(table_data)-1, j].set_facecolor('#D6E4F0')
    
    plt.tight_layout()
    plt.show()

print("Gantt图绘制函数已定义！")

### 算法1：FCFS - 先来先服务 (First Come First Served)

**原理**：按照进程到达就绪队列的先后顺序进行调度。最简单的调度算法。

**特点**：
- **非抢占式 (Non-preemptive)**：一旦进程获得CPU，直到执行完毕才释放
- 实现简单，使用 FIFO 队列
- **缺点**：可能产生 **护航效应 (Convoy Effect)** —— 短进程排在长进程后面，等待时间很长
- 平均等待时间通常较长

In [ ]:
# ========================================
# FCFS 调度算法实现
# ========================================

def fcfs_scheduling(processes):
    """先来先服务调度算法"""
    # 按到达时间排序
    procs = sorted(processes, key=lambda p: (p.arrival_time, p.pid))
    
    schedule = []  # Gantt图数据
    info = {}      # 统计信息
    current_time = 0
    
    for p in procs:
        # 如果CPU空闲（当前时间 < 进程到达时间），需要等待
        if current_time < p.arrival_time:
            schedule.append(('Idle', current_time, p.arrival_time))
            current_time = p.arrival_time
        
        start = current_time
        end = current_time + p.burst_time
        schedule.append((p.pid, start, end))
        
        info[p.pid] = {
            'arrival': p.arrival_time,
            'burst': p.burst_time,
            'finish': end
        }
        current_time = end
    
    return schedule, info

# 运行FCFS
processes = create_sample_processes()
schedule, info = fcfs_scheduling(processes)

print("FCFS 调度顺序：")
for name, start, end in schedule:
    if name != 'Idle':
        print(f"  P{name}: 时间 {start} → {end}")

draw_gantt_chart(schedule, 'FCFS - 先来先服务 (First Come First Served)', info)

### 算法2：SJF - 最短作业优先 (Shortest Job First)

**原理**：选择执行时间（burst time）最短的进程优先执行。

**两种变体**：
- **非抢占式 SJF (Non-preemptive SJF)**：进程一旦开始执行就不会被打断
- **抢占式 SJF / SRTF (Shortest Remaining Time First)**：如果新到达的进程执行时间更短，当前进程会被抢占

**特点**：
- 理论上可以证明 SJF 产生 **最小平均等待时间**
- **缺点**：可能导致 **饥饿 (Starvation)** —— 长进程可能永远得不到执行
- 实际中难以实现，因为无法准确预知进程的执行时间

In [ ]:
# ========================================
# SJF 非抢占式调度算法实现
# ========================================

def sjf_scheduling(processes):
    """最短作业优先调度算法（非抢占式）"""
    procs = [copy.deepcopy(p) for p in processes]
    n = len(procs)
    completed = 0
    current_time = 0
    done = [False] * n
    schedule = []
    info = {}
    
    while completed < n:
        # 找到已到达且未完成的进程中，burst_time最短的
        candidates = [(i, procs[i]) for i in range(n) 
                      if not done[i] and procs[i].arrival_time <= current_time]
        
        if not candidates:
            # 没有可用进程，时间跳到下一个进程到达
            next_arrival = min(p.arrival_time for i, p in enumerate(procs) if not done[i])
            schedule.append(('Idle', current_time, next_arrival))
            current_time = next_arrival
            continue
        
        # 选择burst_time最短的（相同则选到达时间早的）
        idx, proc = min(candidates, key=lambda x: (x[1].burst_time, x[1].arrival_time))
        
        start = current_time
        end = current_time + proc.burst_time
        schedule.append((proc.pid, start, end))
        
        info[proc.pid] = {
            'arrival': proc.arrival_time,
            'burst': proc.burst_time,
            'finish': end
        }
        
        done[idx] = True
        completed += 1
        current_time = end
    
    return schedule, info

# 运行SJF
processes = create_sample_processes()
schedule, info = sjf_scheduling(processes)

print("SJF 调度顺序：")
for name, start, end in schedule:
    if name != 'Idle':
        print(f"  P{name}: 时间 {start} → {end}")

draw_gantt_chart(schedule, 'SJF - 最短作业优先 (Shortest Job First, Non-preemptive)', info)

### 算法3：优先级调度 (Priority Scheduling)

**原理**：每个进程都有一个优先级值，CPU 分配给优先级最高的进程。

**特点**：
- 可以是抢占式或非抢占式
- 优先级可以是内部定义的（基于时间限制、内存需求等）或外部定义的（用户指定）
- **缺点**：同样会导致 **饥饿 (Starvation)**
- **解决方案**：**老化 (Aging)** —— 随着时间推移逐渐增加长时间等待进程的优先级

> **注意**：在我们的例子中，**数字越小 = 优先级越高**（这是常见的约定）

In [ ]:
# ========================================
# 优先级调度算法实现（非抢占式）
# ========================================

def priority_scheduling(processes):
    """优先级调度算法（非抢占式，数字越小优先级越高）"""
    procs = [copy.deepcopy(p) for p in processes]
    n = len(procs)
    completed = 0
    current_time = 0
    done = [False] * n
    schedule = []
    info = {}
    
    while completed < n:
        candidates = [(i, procs[i]) for i in range(n) 
                      if not done[i] and procs[i].arrival_time <= current_time]
        
        if not candidates:
            next_arrival = min(p.arrival_time for i, p in enumerate(procs) if not done[i])
            schedule.append(('Idle', current_time, next_arrival))
            current_time = next_arrival
            continue
        
        # 选择优先级最高（数字最小）的进程
        idx, proc = min(candidates, key=lambda x: (x[1].priority, x[1].arrival_time))
        
        start = current_time
        end = current_time + proc.burst_time
        schedule.append((proc.pid, start, end))
        
        info[proc.pid] = {
            'arrival': proc.arrival_time,
            'burst': proc.burst_time,
            'finish': end
        }
        
        done[idx] = True
        completed += 1
        current_time = end
    
    return schedule, info

# 运行优先级调度
processes = create_sample_processes()
schedule, info = priority_scheduling(processes)

print("Priority 调度顺序（优先级：数字越小越优先）：")
for name, start, end in schedule:
    if name != 'Idle':
        p = [p for p in processes if p.pid == name][0]
        print(f"  P{name} (优先级={p.priority}): 时间 {start} → {end}")

draw_gantt_chart(schedule, 'Priority Scheduling - 优先级调度 (Non-preemptive)', info)

### 算法4：Round Robin - 轮转调度

**原理**：每个进程获得一个固定的 **时间片 (Time Quantum/Time Slice)**，时间片用完后即使没执行完也必须让出 CPU，排到就绪队列末尾。

**特点**：
- **抢占式 (Preemptive)** 算法
- 专为分时系统 (Time-sharing Systems) 设计
- 公平性好，每个进程都能得到 CPU 时间
- 没有饥饿问题

**时间片大小的影响**：
- **时间片太大**：退化为 FCFS
- **时间片太小**：上下文切换开销过大
- 经验法则：时间片应大到足以让大多数进程完成一次 CPU burst

> **考试重点**：Round Robin 是面试和考试中最常出现的调度算法！

In [ ]:
# ========================================
# Round Robin 调度算法实现
# ========================================

def round_robin_scheduling(processes, quantum=2):
    """轮转调度算法"""
    procs = [copy.deepcopy(p) for p in processes]
    n = len(procs)
    remaining = {p.pid: p.burst_time for p in procs}
    arrival = {p.pid: p.arrival_time for p in procs}
    burst = {p.pid: p.burst_time for p in procs}
    
    ready_queue = deque()
    schedule = []
    info = {}
    current_time = 0
    completed = 0
    
    # 按到达时间排序
    sorted_procs = sorted(procs, key=lambda p: (p.arrival_time, p.pid))
    proc_index = 0  # 下一个要检查到达的进程
    
    # 将时间0到达的进程加入队列
    while proc_index < n and sorted_procs[proc_index].arrival_time <= current_time:
        ready_queue.append(sorted_procs[proc_index].pid)
        proc_index += 1
    
    while completed < n:
        if not ready_queue:
            # 队列为空，跳到下一个进程到达
            if proc_index < n:
                next_time = sorted_procs[proc_index].arrival_time
                schedule.append(('Idle', current_time, next_time))
                current_time = next_time
                while proc_index < n and sorted_procs[proc_index].arrival_time <= current_time:
                    ready_queue.append(sorted_procs[proc_index].pid)
                    proc_index += 1
            continue
        
        pid = ready_queue.popleft()
        
        # 执行时间 = min(时间片, 剩余时间)
        exec_time = min(quantum, remaining[pid])
        start = current_time
        current_time += exec_time
        remaining[pid] -= exec_time
        
        schedule.append((pid, start, current_time))
        
        # 检查在执行期间是否有新进程到达
        while proc_index < n and sorted_procs[proc_index].arrival_time <= current_time:
            ready_queue.append(sorted_procs[proc_index].pid)
            proc_index += 1
        
        if remaining[pid] == 0:
            # 进程完成
            completed += 1
            info[pid] = {
                'arrival': arrival[pid],
                'burst': burst[pid],
                'finish': current_time
            }
        else:
            # 进程未完成，放回队列末尾
            ready_queue.append(pid)
    
    return schedule, info

# 运行Round Robin（时间片=2）
processes = create_sample_processes()
schedule, info = round_robin_scheduling(processes, quantum=2)

print("Round Robin 调度顺序（时间片 quantum=2）：")
for name, start, end in schedule:
    if name != 'Idle':
        print(f"  P{name}: 时间 {start} → {end}")

draw_gantt_chart(schedule, 'Round Robin (时间片 Quantum = 2)', info)

### 算法5：多级队列 (Multilevel Queue Scheduling)

**原理**：将就绪队列分为多个独立的队列，每个队列有自己的调度算法。进程根据属性（如优先级、类型）被永久分配到某个队列。

**典型的队列划分**：

| 队列层级 | 类型 | 调度算法 | 说明 |
|----------|------|----------|------|
| 最高优先级 | 系统进程 | Priority | 操作系统核心进程 |
| 高优先级 | 交互式进程 | Round Robin | 用户交互程序，需要快速响应 |
| 中优先级 | 批处理进程 | SJF | 后台批处理任务 |
| 最低优先级 | 学生进程 | FCFS | 低优先级任务 |

**队列之间的调度**：
- **固定优先级**：高优先级队列为空时才执行低优先级队列（可能导致饥饿）
- **时间片分配**：每个队列获得一定比例的 CPU 时间（如前台80%，后台20%）

**多级反馈队列 (Multilevel Feedback Queue)**：
- 进程可以在队列之间移动
- CPU密集型进程会逐渐降到低优先级队列
- I/O密集型进程保持在高优先级队列
- 可以实现老化 (Aging) 来防止饥饿

In [ ]:
# ========================================
# 多级队列可视化
# ========================================

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 12)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('多级队列调度 (Multilevel Queue Scheduling)', fontsize=14, fontweight='bold')

# 队列定义
queues = [
    (8.5, '系统进程\nSystem Processes', '#FF6B6B', 'Priority', ['OS kernel', 'Drivers']),
    (6.5, '交互式进程\nInteractive', '#4ECDC4', 'Round Robin', ['Editor', 'Browser']),
    (4.5, '批处理进程\nBatch', '#45B7D1', 'SJF', ['Compile', 'Backup']),
    (2.5, '低优先级进程\nLow Priority', '#96CEB4', 'FCFS', ['Log cleanup']),
]

for y, label, color, algo, examples in queues:
    # 队列框
    rect = plt.Rectangle((0.5, y - 0.7), 5, 1.2, fill=True, 
                         facecolor=color, edgecolor='black', lw=2, alpha=0.7)
    ax.add_patch(rect)
    ax.text(3, y, label, ha='center', va='center', fontsize=10, fontweight='bold')
    
    # 调度算法
    ax.text(6.5, y, f'算法: {algo}', ha='left', va='center', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='gray'))
    
    # 示例进程
    ax.text(9.5, y, f'例: {", ".join(examples)}', ha='left', va='center', fontsize=9, color='gray')

# 优先级箭头
ax.annotate('', xy=(0.2, 2.0), xytext=(0.2, 9.0),
            arrowprops=dict(arrowstyle='->', lw=3, color='red'))
ax.text(0.15, 5.5, '优\n先\n级\n递\n减', ha='center', va='center', fontsize=10, 
        color='red', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ========================================
# 所有调度算法对比
# ========================================

processes = create_sample_processes()

# 收集每种算法的平均等待时间和平均周转时间
algorithms = {}

for name, func, args in [
    ('FCFS', fcfs_scheduling, {}),
    ('SJF', sjf_scheduling, {}),
    ('Priority', priority_scheduling, {}),
    ('RR(q=2)', round_robin_scheduling, {'quantum': 2}),
    ('RR(q=3)', round_robin_scheduling, {'quantum': 3}),
]:
    procs = create_sample_processes()
    schedule, info = func(procs, **args) if args else func(procs)
    
    avg_turnaround = np.mean([info[pid]['finish'] - info[pid]['arrival'] for pid in info])
    avg_waiting = np.mean([info[pid]['finish'] - info[pid]['arrival'] - info[pid]['burst'] for pid in info])
    algorithms[name] = (avg_waiting, avg_turnaround)

# 绘制对比图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

names = list(algorithms.keys())
waiting_times = [algorithms[n][0] for n in names]
turnaround_times = [algorithms[n][1] for n in names]

colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF99FF']

bars1 = ax1.bar(names, waiting_times, color=colors, edgecolor='black')
ax1.set_title('平均等待时间对比\n(Average Waiting Time)', fontsize=13, fontweight='bold')
ax1.set_ylabel('时间 (Time Units)')
for bar, val in zip(bars1, waiting_times):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             f'{val:.1f}', ha='center', fontweight='bold')

bars2 = ax2.bar(names, turnaround_times, color=colors, edgecolor='black')
ax2.set_title('平均周转时间对比\n(Average Turnaround Time)', fontsize=13, fontweight='bold')
ax2.set_ylabel('时间 (Time Units)')
for bar, val in zip(bars2, turnaround_times):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             f'{val:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n观察结论：")
print("- SJF 通常有最短的平均等待时间（这是理论可证明的）")
print("- FCFS 简单但效率可能最低")
print("- Round Robin 的性能取决于时间片大小")
print("- Priority 的效果取决于优先级分配是否合理")

---
## 第二部分：内存管理 (Memory Management)

### 2.1 为什么需要内存管理？

内存（RAM）是有限的资源，操作系统需要：
- 将多个进程同时加载到内存中（多道程序设计）
- 保护每个进程的内存空间不被其他进程访问
- 当物理内存不足时，提供虚拟内存机制

### 2.2 分页 (Paging)

**分页**是最常用的内存管理方案：

- 将 **物理内存 (Physical Memory)** 分成大小相同的块，称为 **帧 (Frame)**
- 将 **逻辑地址空间 (Logical Address Space)** 分成同样大小的块，称为 **页 (Page)**
- 页和帧的大小相同（通常为 4KB）
- 操作系统维护 **页表 (Page Table)** 来记录页到帧的映射关系

**地址转换 (Address Translation)**：
- 逻辑地址 = **页号 (Page Number)** + **页内偏移 (Offset)**
- 物理地址 = **帧号 (Frame Number)** + **页内偏移 (Offset)**

**优点**：
- 消除了外部碎片 (External Fragmentation)
- 进程的页不需要连续存放在物理内存中

**缺点**：
- 可能产生内部碎片 (Internal Fragmentation)——最后一页可能未被完全使用
- 页表本身需要占用内存空间

In [ ]:
# ========================================
# 分页机制可视化
# ========================================

fig, axes = plt.subplots(1, 3, figsize=(16, 8))

# 左：逻辑地址空间（页）
ax1 = axes[0]
ax1.set_title('逻辑地址空间\n(Logical Address Space)', fontsize=12, fontweight='bold')
ax1.set_xlim(0, 4)
ax1.set_ylim(0, 8)
ax1.axis('off')

page_colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF99FF', '#FFFF99']
pages = ['页0 (Page 0)', '页1 (Page 1)', '页2 (Page 2)', 
         '页3 (Page 3)', '页4 (Page 4)', '页5 (Page 5)']

for i, (page, color) in enumerate(zip(pages, page_colors)):
    y = 7 - i * 1.1
    rect = plt.Rectangle((0.5, y - 0.4), 3, 0.8, fill=True,
                         facecolor=color, edgecolor='black', lw=1.5)
    ax1.add_patch(rect)
    ax1.text(2, y, page, ha='center', va='center', fontsize=9, fontweight='bold')

# 中：页表
ax2 = axes[1]
ax2.set_title('页表\n(Page Table)', fontsize=12, fontweight='bold')
ax2.axis('off')

# 页到帧的映射
page_to_frame = {0: 3, 1: 7, 2: 1, 3: 5, 4: 0, 5: 4}

table_data = [['页号\nPage#', '帧号\nFrame#', '有效位\nValid']]
for p in range(6):
    table_data.append([str(p), str(page_to_frame[p]), '1'])

table = ax2.table(cellText=table_data, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 2.0)
for j in range(3):
    table[0, j].set_facecolor('#4472C4')
    table[0, j].set_text_props(color='white', fontweight='bold')

# 右：物理内存（帧）
ax3 = axes[2]
ax3.set_title('物理内存\n(Physical Memory)', fontsize=12, fontweight='bold')
ax3.set_xlim(0, 4)
ax3.set_ylim(0, 10)
ax3.axis('off')

frame_to_page = {v: k for k, v in page_to_frame.items()}

for i in range(8):
    y = 9 - i * 1.1
    if i in frame_to_page:
        page_idx = frame_to_page[i]
        color = page_colors[page_idx]
        label = f'帧{i}: 页{page_idx}'
    else:
        color = '#EEEEEE'
        label = f'帧{i}: 空闲'
    
    rect = plt.Rectangle((0.5, y - 0.4), 3, 0.8, fill=True,
                         facecolor=color, edgecolor='black', lw=1.5)
    ax3.add_patch(rect)
    ax3.text(2, y, label, ha='center', va='center', fontsize=9, fontweight='bold')

plt.suptitle('分页内存管理示意图', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n注意观察：逻辑上连续的页在物理内存中可以不连续存放！")
print("这就是分页的核心优势 —— 消除了外部碎片。")

### 2.3 分段 (Segmentation)

**分段**将程序按逻辑单元划分：

| 段 (Segment) | 内容示例 |
|:-------------|:---------|
| 代码段 (Code) | 程序的可执行指令 |
| 数据段 (Data) | 全局变量 |
| 栈段 (Stack) | 函数调用栈 |
| 堆段 (Heap) | 动态分配的内存 |

**与分页的区别**：

| 特性 | 分页 (Paging) | 分段 (Segmentation) |
|------|:-------------|:-------------------|
| 大小 | 固定大小 | 可变大小 |
| 对用户可见？ | 不可见（对程序员透明） | 可见（程序员意识到段的存在） |
| 地址结构 | 页号 + 偏移 | 段号 + 偏移 |
| 碎片类型 | 内部碎片 | 外部碎片 |
| 逻辑意义 | 无逻辑意义 | 有逻辑意义（代码/数据/栈） |
| 共享 | 不方便 | 方便（可共享代码段） |

> 现代操作系统通常结合使用分页和分段：**段页式 (Segmented Paging)**。

### 2.4 虚拟内存 (Virtual Memory)

**虚拟内存**允许程序使用比实际物理内存更大的地址空间。

**核心思想**：
- 不需要将整个程序都加载到物理内存中
- 只将当前需要的页加载到内存
- 不需要的页存放在磁盘上（**交换空间/页面文件 Swap Space/Page File**）

**按需分页 (Demand Paging)**：
1. 进程访问某个页
2. 检查页表，如果该页在内存中（有效位=1），正常访问
3. 如果该页不在内存中（有效位=0），产生 **缺页中断 (Page Fault)**
4. 操作系统从磁盘加载该页到空闲帧
5. 更新页表
6. 重新执行导致缺页的指令

**抖动 (Thrashing)**：
- 当系统中进程太多，每个进程分到的帧太少
- 频繁发生缺页中断
- CPU 大部分时间在处理缺页中断，有效利用率极低
- 解决方案：减少多道程序的度 (degree of multiprogramming)

### 2.5 页面置换算法 (Page Replacement Algorithms)

当发生缺页中断且物理内存已满时，需要选择一个页换出。常见算法：

| 算法 | 英文 | 策略 | 特点 |
|------|------|------|------|
| FIFO | First In First Out | 替换最早进入内存的页 | 简单但可能不是最优的，可能出现 Belady 异常 |
| LRU | Least Recently Used | 替换最近最少使用的页 | 效果好，但实现开销较大 |
| 最优 | Optimal (OPT) | 替换将来最长时间不会被使用的页 | 理论最优，但需要预知未来，不可实现 |

In [ ]:
# ========================================
# 页面置换算法模拟器
# ========================================

def fifo_page_replacement(pages, num_frames):
    """FIFO 页面置换算法"""
    frames = []          # 当前在内存中的页
    queue = deque()      # FIFO队列
    page_faults = 0
    history = []         # 记录每一步的状态
    
    for page in pages:
        fault = False
        if page not in frames:
            fault = True
            page_faults += 1
            if len(frames) >= num_frames:
                # 移除最早进入的页
                old = queue.popleft()
                frames.remove(old)
            frames.append(page)
            queue.append(page)
        
        history.append((page, list(frames), fault))
    
    return page_faults, history

def lru_page_replacement(pages, num_frames):
    """LRU 页面置换算法"""
    frames = []
    page_faults = 0
    history = []
    
    for i, page in enumerate(pages):
        fault = False
        if page not in frames:
            fault = True
            page_faults += 1
            if len(frames) >= num_frames:
                # 找到最近最少使用的页
                lru_page = None
                lru_time = float('inf')
                for f in frames:
                    # 向前查找最后一次使用的时间
                    for j in range(i - 1, -1, -1):
                        if pages[j] == f:
                            if j < lru_time:
                                lru_time = j
                                lru_page = f
                            break
                frames.remove(lru_page)
            frames.append(page)
        else:
            # 页已在内存中，更新使用顺序（移到最后）
            frames.remove(page)
            frames.append(page)
        
        history.append((page, list(frames), fault))
    
    return page_faults, history

def optimal_page_replacement(pages, num_frames):
    """最优页面置换算法"""
    frames = []
    page_faults = 0
    history = []
    
    for i, page in enumerate(pages):
        fault = False
        if page not in frames:
            fault = True
            page_faults += 1
            if len(frames) >= num_frames:
                # 找到将来最晚使用（或不再使用）的页
                farthest = -1
                victim = frames[0]
                for f in frames:
                    try:
                        next_use = pages[i+1:].index(f)
                    except ValueError:
                        victim = f
                        break
                    if next_use > farthest:
                        farthest = next_use
                        victim = f
                frames.remove(victim)
            frames.append(page)
        
        history.append((page, list(frames), fault))
    
    return page_faults, history

print("页面置换算法模拟器已定义！")

In [ ]:
# ========================================
# 页面置换算法演示与可视化
# ========================================

# 页面引用序列
page_sequence = [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2, 1, 2, 0, 1, 7, 0, 1]
num_frames = 3

print(f"页面引用序列 (Reference String): {page_sequence}")
print(f"帧数 (Number of Frames): {num_frames}")
print("=" * 70)

algorithms_pr = [
    ('FIFO', fifo_page_replacement),
    ('LRU', lru_page_replacement),
    ('Optimal', optimal_page_replacement),
]

fig, axes = plt.subplots(3, 1, figsize=(18, 12))

fault_counts = {}

for idx, (name, func) in enumerate(algorithms_pr):
    faults, history = func(page_sequence, num_frames)
    fault_counts[name] = faults
    
    ax = axes[idx]
    n = len(page_sequence)
    
    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(-0.5, num_frames + 1.5)
    ax.set_title(f'{name} 页面置换 (缺页次数 Page Faults = {faults})', 
                 fontsize=12, fontweight='bold')
    ax.set_xticks(range(n))
    ax.set_xticklabels([str(p) for p in page_sequence])
    ax.set_xlabel('访问的页号 (Page Referenced)')
    ax.set_yticks(range(num_frames))
    ax.set_yticklabels([f'帧{i}' for i in range(num_frames)])
    
    for step, (page, frames_state, fault) in enumerate(history):
        # 顶部标记缺页
        if fault:
            ax.text(step, num_frames + 0.5, 'F', ha='center', va='center',
                    fontsize=10, color='red', fontweight='bold')
        else:
            ax.text(step, num_frames + 0.5, 'H', ha='center', va='center',
                    fontsize=10, color='green', fontweight='bold')
        
        # 绘制帧内容
        for frame_idx in range(num_frames):
            if frame_idx < len(frames_state):
                p = frames_state[frame_idx]
                color = '#FF9999' if fault and p == page else '#AED6F1'
                rect = plt.Rectangle((step - 0.4, frame_idx - 0.35), 0.8, 0.7,
                                     facecolor=color, edgecolor='black', lw=1)
                ax.add_patch(rect)
                ax.text(step, frame_idx, str(p), ha='center', va='center',
                        fontsize=11, fontweight='bold')
    
    ax.axhline(y=num_frames - 0.05, color='gray', linestyle='--', alpha=0.3)
    
    # 图例说明
    ax.text(n - 0.5, num_frames + 1.2, 'F=缺页Fault  H=命中Hit', 
            ha='right', va='center', fontsize=9, fontstyle='italic')

plt.tight_layout()
plt.show()

print("\n缺页次数对比：")
for name, count in fault_counts.items():
    rate = count / len(page_sequence) * 100
    print(f"  {name:8s}: {count} 次缺页 (缺页率 = {rate:.1f}%)")
print(f"\nOptimal 是理论下界 —— 任何可实现的算法缺页次数都 ≥ {fault_counts['Optimal']}")

---
## 第三部分：死锁 (Deadlock)

### 3.1 什么是死锁？

**死锁 (Deadlock)**：两个或多个进程互相等待对方持有的资源，导致所有进程都无法继续执行。

**经典比喻**：两辆车在单行桥上相向而行，都不愿意后退，结果谁也过不去。

### 3.2 死锁的四个必要条件 (Coffman Conditions)

死锁发生必须**同时**满足以下四个条件：

| 条件 | 英文 | 说明 |
|------|------|------|
| 互斥 | **Mutual Exclusion** | 资源不能被多个进程同时使用 |
| 持有并等待 | **Hold and Wait** | 进程持有至少一个资源的同时等待获取其他资源 |
| 非抢占 | **No Preemption** | 资源不能被强制从进程中夺走，只能由持有者主动释放 |
| 循环等待 | **Circular Wait** | 存在一组等待进程形成环路：P1等P2，P2等P3，...，Pn等P1 |

> **考试重点**：这四个条件必须全部背下来！只要破坏其中任何一个条件，就可以预防死锁。

In [ ]:
# ========================================
# 死锁可视化 - 资源分配图
# ========================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# 左图：有死锁的情况
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)
ax1.axis('off')
ax1.set_title('有死锁的资源分配图\n(Deadlocked State)', fontsize=12, fontweight='bold', color='red')

# 进程（圆形）
for x, y, label in [(2, 7, 'P1'), (8, 7, 'P2'), (5, 2, 'P3')]:
    circle = plt.Circle((x, y), 0.8, color='#AED6F1', ec='black', lw=2)
    ax1.add_patch(circle)
    ax1.text(x, y, label, ha='center', va='center', fontsize=12, fontweight='bold')

# 资源（方形）
for x, y, label in [(5, 8.5, 'R1'), (2, 3.5, 'R2'), (8, 3.5, 'R3')]:
    rect = plt.Rectangle((x-0.6, y-0.5), 1.2, 1, color='#F9E79F', ec='black', lw=2)
    ax1.add_patch(rect)
    ax1.text(x, y, label, ha='center', va='center', fontsize=12, fontweight='bold')

# 箭头：进程→资源 = 请求，资源→进程 = 已分配
# P1持有R2，请求R1
ax1.annotate('', xy=(4.4, 8.5), xytext=(2.7, 7.5), arrowprops=dict(arrowstyle='->', lw=2, color='blue'))
ax1.annotate('', xy=(2, 6.2), xytext=(2, 4.5), arrowprops=dict(arrowstyle='->', lw=2, color='red'))
# P2持有R1，请求R3
ax1.annotate('', xy=(8, 4.5), xytext=(8, 6.2), arrowprops=dict(arrowstyle='->', lw=2, color='blue'))
ax1.annotate('', xy=(7.3, 7), xytext=(5.6, 8.2), arrowprops=dict(arrowstyle='->', lw=2, color='red'))
# P3持有R3，请求R2
ax1.annotate('', xy=(2.6, 3.5), xytext=(4.2, 2.3), arrowprops=dict(arrowstyle='->', lw=2, color='blue'))
ax1.annotate('', xy=(5.5, 2.5), xytext=(7.4, 3.2), arrowprops=dict(arrowstyle='->', lw=2, color='red'))

# 图例
ax1.plot([], [], 'b->', label='请求 (Request)', lw=2)
ax1.plot([], [], 'r->', label='已分配 (Assigned)', lw=2)
ax1.legend(loc='lower left', fontsize=10)
ax1.text(5, 0.3, '循环等待: P1→R1→P2→R3→P3→R2→P1', 
         ha='center', va='center', fontsize=10, color='red', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='lightyellow'))

# 右图：无死锁
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('无死锁的资源分配图\n(Safe State)', fontsize=12, fontweight='bold', color='green')

for x, y, label in [(3, 7, 'P1'), (7, 7, 'P2')]:
    circle = plt.Circle((x, y), 0.8, color='#AED6F1', ec='black', lw=2)
    ax2.add_patch(circle)
    ax2.text(x, y, label, ha='center', va='center', fontsize=12, fontweight='bold')

for x, y, label in [(3, 3.5, 'R1'), (7, 3.5, 'R2')]:
    rect = plt.Rectangle((x-0.6, y-0.5), 1.2, 1, color='#F9E79F', ec='black', lw=2)
    ax2.add_patch(rect)
    ax2.text(x, y, label, ha='center', va='center', fontsize=12, fontweight='bold')

# P1持有R1，请求R2
ax2.annotate('', xy=(6.4, 3.5), xytext=(3.8, 7), arrowprops=dict(arrowstyle='->', lw=2, color='blue'))
ax2.annotate('', xy=(3, 6.2), xytext=(3, 4.5), arrowprops=dict(arrowstyle='->', lw=2, color='red'))
# P2持有R2（不请求其他资源，所以P2可以完成并释放R2）
ax2.annotate('', xy=(7, 6.2), xytext=(7, 4.5), arrowprops=dict(arrowstyle='->', lw=2, color='red'))

ax2.plot([], [], 'b->', label='请求 (Request)', lw=2)
ax2.plot([], [], 'r->', label='已分配 (Assigned)', lw=2)
ax2.legend(loc='lower left', fontsize=10)
ax2.text(5, 0.3, 'P2可以完成→释放R2→P1获得R2→无死锁', 
         ha='center', va='center', fontsize=10, color='green', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='lightyellow'))

plt.tight_layout()
plt.show()

### 3.3 死锁的处理策略

#### 策略1：死锁预防 (Deadlock Prevention)

通过破坏四个必要条件之一来预防死锁：

| 破坏条件 | 方法 | 缺点 |
|----------|------|------|
| 互斥 | 使资源可共享（如只读文件） | 某些资源本身不可共享（如打印机） |
| 持有并等待 | 进程必须一次性申请所有资源 | 资源利用率低，可能饥饿 |
| 非抢占 | 允许强制回收资源 | 可能导致工作丢失 |
| 循环等待 | 对资源编号，按序申请 | 限制了资源请求的灵活性 |

#### 策略2：死锁避免 (Deadlock Avoidance) - 银行家算法 (Banker's Algorithm)

在分配资源前检查是否会导致不安全状态。

In [ ]:
# ========================================
# 银行家算法实现 (Banker's Algorithm)
# ========================================

def bankers_algorithm(available, maximum, allocation):
    """
    银行家算法 - 判断系统是否处于安全状态
    
    参数:
        available: 各类资源的可用数量 [A, B, C, ...]
        maximum: 每个进程对每种资源的最大需求
        allocation: 每个进程当前已分配的资源
    
    返回:
        (是否安全, 安全序列)
    """
    n = len(maximum)       # 进程数
    m = len(available)     # 资源类型数
    
    # 计算 Need 矩阵: Need[i][j] = Maximum[i][j] - Allocation[i][j]
    need = [[maximum[i][j] - allocation[i][j] for j in range(m)] for i in range(n)]
    
    print("=" * 60)
    print("银行家算法分析 (Banker's Algorithm Analysis)")
    print("=" * 60)
    
    # 打印当前状态
    print(f"\n可用资源 Available: {available}")
    print(f"\n{'进程':^6}{'最大需求 Max':^18}{'已分配 Alloc':^18}{'还需要 Need':^18}")
    print("-" * 60)
    for i in range(n):
        print(f"{'P'+str(i):^6}{str(maximum[i]):^18}{str(allocation[i]):^18}{str(need[i]):^18}")
    
    # 安全性算法
    work = available.copy()
    finish = [False] * n
    safe_sequence = []
    
    print(f"\n--- 开始安全性检查 ---")
    
    for round_num in range(n):
        found = False
        for i in range(n):
            if not finish[i]:
                # 检查 Need[i] <= Work
                if all(need[i][j] <= work[j] for j in range(m)):
                    print(f"\n第{round_num+1}轮: P{i} 可以执行")
                    print(f"  Need[P{i}] = {need[i]} <= Work = {work} ✓")
                    
                    # 模拟执行并释放资源
                    work = [work[j] + allocation[i][j] for j in range(m)]
                    print(f"  P{i} 完成后释放资源，Work 变为: {work}")
                    
                    finish[i] = True
                    safe_sequence.append(i)
                    found = True
                    break
        
        if not found:
            break
    
    is_safe = all(finish)
    
    print(f"\n{'='*60}")
    if is_safe:
        seq_str = ' → '.join([f'P{i}' for i in safe_sequence])
        print(f"结论: 系统处于安全状态 (SAFE STATE)! ✓")
        print(f"安全序列 (Safe Sequence): {seq_str}")
    else:
        print(f"结论: 系统处于不安全状态 (UNSAFE STATE)! ✗")
        print(f"无法找到安全序列，可能发生死锁！")
    
    return is_safe, safe_sequence

# 示例：5个进程，3种资源 (A, B, C)
print("示例场景：5个进程，3种资源 (A, B, C)")
print("资源总量: A=10, B=5, C=7\n")

available = [3, 3, 2]    # 当前可用
maximum = [
    [7, 5, 3],   # P0
    [3, 2, 2],   # P1
    [9, 0, 2],   # P2
    [2, 2, 2],   # P3
    [4, 3, 3],   # P4
]
allocation = [
    [0, 1, 0],   # P0
    [2, 0, 0],   # P1
    [3, 0, 2],   # P2
    [2, 1, 1],   # P3
    [0, 0, 2],   # P4
]

is_safe, sequence = bankers_algorithm(available, maximum, allocation)

### 3.4 死锁检测与恢复 (Deadlock Detection and Recovery)

如果不预防也不避免死锁，系统可以允许死锁发生，然后检测并恢复。

**检测方法**：
- 维护资源分配图，定期检测是否存在循环等待
- 使用类似银行家算法的方法检测

**恢复方法**：
1. **终止进程 (Process Termination)**
   - 终止所有死锁进程（代价大）
   - 逐一终止直到死锁解除（需要决定终止顺序）

2. **资源抢占 (Resource Preemption)**
   - 从某个进程中回收资源
   - 需要考虑：选择哪个进程？如何回滚？如何避免饥饿？

---
## 第四部分：文件系统 (File Systems)

### 4.1 文件系统的功能

文件系统负责管理磁盘上数据的存储、组织和访问：
- 文件的创建、删除、读写
- 目录管理（层次结构）
- 空间分配和回收
- 访问控制（权限）
- 数据完整性保护

### 4.2 三种文件系统对比

| 特性 | FAT (FAT16/FAT32) | NTFS | ext4 |
|------|:-------------------|:-----|:-----|
| 全称 | File Allocation Table | New Technology File System | Fourth Extended Filesystem |
| 开发者 | Microsoft | Microsoft | Linux 社区 |
| 使用场景 | U盘、SD卡、旧系统 | Windows系统盘 | Linux系统盘 |
| 最大文件大小 | FAT32: 4GB | 16TB (理论256TB) | 16TB |
| 最大分区大小 | FAT32: 2TB | 256TB | 1EB |
| 日志功能 | 无 | 有 (Journaling) | 有 (Journaling) |
| 权限系统 | 无 | ACL (访问控制列表) | Unix权限 + ACL |
| 加密 | 无 | 支持 (EFS/BitLocker) | 支持 |
| 碎片问题 | 严重 | 较少 | 很少 |
| 兼容性 | 几乎所有OS | 主要Windows | 主要Linux |

### 4.3 文件分配方式

| 方式 | 英文 | 原理 | 优点 | 缺点 |
|------|------|------|------|------|
| 连续分配 | Contiguous | 文件占用连续的磁盘块 | 顺序/随机访问快 | 外部碎片、文件大小难以扩展 |
| 链接分配 | Linked | 每个块包含指向下一块的指针 | 无外部碎片 | 只能顺序访问、指针占空间 |
| 索引分配 | Indexed | 用索引块记录所有数据块位置 | 支持随机访问 | 索引块占额外空间 |

In [ ]:
# ========================================
# 文件分配方式可视化
# ========================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 连续分配
ax1 = axes[0]
ax1.set_title('连续分配\n(Contiguous Allocation)', fontsize=12, fontweight='bold')
ax1.set_xlim(0, 5)
ax1.set_ylim(0, 10)
ax1.axis('off')

for i in range(8):
    y = 8.5 - i
    if 2 <= i <= 5:  # 文件A占用块2-5
        color = '#FF9999'
        label = f'块{i}: 文件A'
    else:
        color = '#EEEEEE'
        label = f'块{i}: 空闲'
    rect = plt.Rectangle((0.5, y - 0.35), 3.5, 0.7, facecolor=color, edgecolor='black', lw=1.5)
    ax1.add_patch(rect)
    ax1.text(2.25, y, label, ha='center', va='center', fontsize=9)

ax1.text(2.25, 9.3, '起始块=2, 长度=4', ha='center', va='center', fontsize=10, color='red')

# 链接分配
ax2 = axes[1]
ax2.set_title('链接分配\n(Linked Allocation)', fontsize=12, fontweight='bold')
ax2.set_xlim(0, 5)
ax2.set_ylim(0, 10)
ax2.axis('off')

file_blocks = {1: 4, 4: 6, 6: 2, 2: -1}  # 块号: 下一块
for i in range(8):
    y = 8.5 - i
    if i in file_blocks:
        color = '#66B2FF'
        nxt = file_blocks[i]
        ptr = f'→{nxt}' if nxt != -1 else '→END'
        label = f'块{i}: 数据 {ptr}'
    else:
        color = '#EEEEEE'
        label = f'块{i}: 空闲'
    rect = plt.Rectangle((0.5, y - 0.35), 3.5, 0.7, facecolor=color, edgecolor='black', lw=1.5)
    ax2.add_patch(rect)
    ax2.text(2.25, y, label, ha='center', va='center', fontsize=9)

ax2.text(2.25, 9.3, '起始块=1, 链: 1→4→6→2', ha='center', va='center', fontsize=10, color='blue')

# 索引分配
ax3 = axes[2]
ax3.set_title('索引分配\n(Indexed Allocation)', fontsize=12, fontweight='bold')
ax3.set_xlim(0, 5)
ax3.set_ylim(0, 10)
ax3.axis('off')

index_block = 3  # 索引块
data_blocks = {1, 4, 6, 2}  # 数据块
for i in range(8):
    y = 8.5 - i
    if i == index_block:
        color = '#99FF99'
        label = f'块{i}: 索引[1,4,6,2]'
    elif i in data_blocks:
        color = '#FFCC99'
        label = f'块{i}: 数据'
    else:
        color = '#EEEEEE'
        label = f'块{i}: 空闲'
    rect = plt.Rectangle((0.5, y - 0.35), 3.5, 0.7, facecolor=color, edgecolor='black', lw=1.5)
    ax3.add_patch(rect)
    ax3.text(2.25, y, label, ha='center', va='center', fontsize=9)

ax3.text(2.25, 9.3, '索引块=3, 数据块=[1,4,6,2]', ha='center', va='center', fontsize=10, color='green')

plt.tight_layout()
plt.show()

---
## 第五部分：综合练习 (Practice Exercises)

### 练习1：调度算法计算题

给定以下进程信息：

| 进程 | 到达时间 | 执行时间 | 优先级 |
|------|----------|----------|--------|
| P1 | 0 | 8 | 2 |
| P2 | 1 | 3 | 1 |
| P3 | 2 | 5 | 3 |
| P4 | 4 | 2 | 4 |

分别使用 FCFS、SJF（非抢占）、Priority（非抢占）和 Round Robin（quantum=3）来调度这些进程。

对于每种算法，画出 Gantt 图并计算：
1. 每个进程的周转时间和等待时间
2. 平均周转时间和平均等待时间

In [ ]:
# 练习1 - 使用模拟器验证你的答案

exercise_processes = [
    Process('1', arrival_time=0, burst_time=8, priority=2),
    Process('2', arrival_time=1, burst_time=3, priority=1),
    Process('3', arrival_time=2, burst_time=5, priority=3),
    Process('4', arrival_time=4, burst_time=2, priority=4),
]

print("请先在纸上手动计算，然后运行下面的代码验证！")
print("(First try to work it out on paper, then verify with the code below)")
print()

for algo_name, algo_func, kwargs in [
    ('FCFS', fcfs_scheduling, {}),
    ('SJF (Non-preemptive)', sjf_scheduling, {}),
    ('Priority (Non-preemptive)', priority_scheduling, {}),
    ('Round Robin (q=3)', round_robin_scheduling, {'quantum': 3}),
]:
    procs = [
        Process('1', 0, 8, 2),
        Process('2', 1, 3, 1),
        Process('3', 2, 5, 3),
        Process('4', 4, 2, 4),
    ]
    schedule, info = algo_func(procs, **kwargs) if kwargs else algo_func(procs)
    draw_gantt_chart(schedule, algo_name, info)

### 练习2：页面置换

给定页面引用序列：`1, 2, 3, 4, 1, 2, 5, 1, 2, 3, 4, 5`

帧数 = 3

分别用 FIFO、LRU、Optimal 算法计算缺页次数。

In [ ]:
# 练习2 - 验证你的答案

ex_pages = [1, 2, 3, 4, 1, 2, 5, 1, 2, 3, 4, 5]
ex_frames = 3

print(f"页面引用序列: {ex_pages}")
print(f"帧数: {ex_frames}")
print()

for name, func in [('FIFO', fifo_page_replacement), 
                    ('LRU', lru_page_replacement), 
                    ('Optimal', optimal_page_replacement)]:
    faults, history = func(ex_pages, ex_frames)
    print(f"{name:10s}: {faults} 次缺页 (Page Faults)")
    # 详细过程
    for page, frames_state, fault in history:
        status = "FAULT" if fault else "HIT  "
        print(f"  访问页{page} → {status} → 帧内容: {frames_state}")
    print()

### 练习3：银行家算法

系统有3种资源 A, B, C，总量分别为 [10, 5, 7]。当前状态如下：

| 进程 | 已分配 (A,B,C) | 最大需求 (A,B,C) |
|------|:-------------|:---------------|
| P0 | (0,1,0) | (7,5,3) |
| P1 | (3,0,2) | (3,2,2) |
| P2 | (3,0,2) | (9,0,2) |
| P3 | (2,1,1) | (2,2,2) |
| P4 | (0,0,2) | (4,3,3) |

问：如果 P1 请求资源 (1,0,2)，应该分配吗？

In [ ]:
# 练习3 - 银行家算法验证

print("=" * 60)
print("当前状态检查：")
available = [3, 3, 2]
maximum = [[7,5,3], [3,2,2], [9,0,2], [2,2,2], [4,3,3]]
allocation = [[0,1,0], [3,0,2], [3,0,2], [2,1,1], [0,0,2]]
is_safe, seq = bankers_algorithm(available, maximum, allocation)

print("\n" + "=" * 60)
print("\n假设分配 P1 的请求 (1,0,2) 后的状态检查：")
# P1 请求 (1,0,2)
# 首先检查：请求 <= Need[P1]? (1,0,2) <= (0,2,0)? 不满足！
# 实际上 Need[P1] = Max[P1] - Alloc[P1] = (3,2,2) - (3,0,2) = (0,2,0)
# (1,0,2) > (0,2,0)，所以请求无效（超过了还需要的量）
print("\nNeed[P1] = Max[P1] - Allocation[P1] = (3,2,2) - (3,0,2) = (0,2,0)")
print("请求 (1,0,2) > Need (0,2,0)？")
print("A: 1 > 0 → 是！请求无效，进程请求超过了其声明的最大需求！")
print("\n让我们改为测试 P1 请求 (0,2,0)：")

new_available = [3, 1, 2]  # available - request
new_allocation = [[0,1,0], [3,2,2], [3,0,2], [2,1,1], [0,0,2]]
is_safe2, seq2 = bankers_algorithm(new_available, maximum, new_allocation)

### 练习4：简答题

1. 解释分页 (Paging) 和分段 (Segmentation) 的区别。(4分)

2. 列出死锁的四个必要条件，并对每个条件说明如何破坏它来预防死锁。(8分)

3. 比较 FCFS 和 Round Robin 调度算法的优缺点。解释为什么 Round Robin 更适合分时系统。(6分)

4. 什么是 thrashing（抖动）？它是如何产生的？如何解决？(4分)

5. 解释 LRU 页面置换算法的工作原理。为什么它通常比 FIFO 效果更好？(4分)

---

### 本章小结 (Chapter Summary)

| 主题 | 关键概念 |
|------|----------|
| 进程管理 | 五状态模型、PCB、上下文切换、FCFS/SJF/Priority/RR/多级队列 |
| 内存管理 | 分页(固定大小)、分段(逻辑单元)、虚拟内存、按需分页 |
| 页面置换 | FIFO、LRU、Optimal |
| 死锁 | 四个必要条件、预防、避免(银行家算法)、检测与恢复 |
| 文件系统 | FAT(简单/兼容好)、NTFS(Windows/功能强)、ext4(Linux/高效) |